<a href="https://colab.research.google.com/github/ragulsulochana/Ragul-codeboosters-2026/blob/main/DAY%208/DAY_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q



In [ ]:
!pip install sentence-transformers chromadb groq pandas
import pandas as pd

import chromadb

from sentence_transformers import SentenceTransformer

from groq import Groq

import os

print("All libraries imported")

All libraries imported


In [ ]:
df = pd.read_csv('/content/college_notes (1).csv')
print("shape of dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("")

shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']



In [ ]:
print("Sabjects in the dataset:")

print(df['subject'].value_counts())

print("Sample of topics:")

print(df[['note_id', 'subject', 'topic']].to_string(index=False))

df['content_length'] = df['content'].apply(len)
print("\nLength of content (number of characters) for each note:")

print(df[['topic', 'content_length']].to_string(index=False))

Sabjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python 

In [ ]:
documents = df['content'].tolist()

ids = [f'note_{row["note_id"]}' for row in df.to_dict("records")]

metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")

print(f"First document ID: {ids[0]}")

print(f"First metadata: {metadatas[0]}")

print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
First document ID: note_N001
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("Loading embedding model...")

print("(This may take 30-60 seconds on first run model is being downloaded)")

print("(Subsequent runs will be faster as the model is cached)")

embedding_model = SentenceTransformer("all-MINILM-L6-v2") # Corrected syntax and typo

print("Embedding model loaded successfully.") # Corrected typos

test_embedding = embedding_model.encode("This is a test sentence.") # Corrected typos and assignment

print(f"Test embedding shape: {test_embedding.shape}") # Corrected typos and f-string usage

print(f"First 5 values of test embedding: {test_embedding[:5]}") # Corrected typos and f-string usage

Loading embedding model...
(This may take 30-60 seconds on first run model is being downloaded)
(Subsequent runs will be faster as the model is cached)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Test embedding shape: (384,)
First 5 values of test embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [ ]:
# Initialize ChromaDB client
chroma_client = chromadb.Client()

# A collection in ChromaDB is like a table in a regular database.
# It groups related documents together.

# The name we give this collection
collection_name = "college_notes_rag"

# Get or create the collection
collection = chroma_client.get_or_create_collection(name=collection_name)

print("Chroma client created.")
print(f"Collection name: {collection_name}")
print(f"Documents in collection so far: {collection.count()}")

Chroma client created.
Collection name: college_notes_rag
Documents in collection so far: 0


In [ ]:
print("Generating embeddings for all 15 notes...")

print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    # The actual text content of each note
    embeddings=embeddings_list,
    # The vector representation of each note
    ids=ids,
    # Unique string IDs like 'note_1', 'note_2'
    metadatas=metadatas
    # Subject and topic info for each note
)

print("Documents successfully added to collection.")

print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)
Documents successfully added to collection.
Total documents in collection: 15


In [ ]:
def retrieve_relevant_chunks(question, top_k=3):

    question_embedding = embedding_model.encode(question).tolist()


    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        include=['documents', 'distances', 'metadatas']
    )

    return results

    print(f"retrieval function defined successfuly")
    print(f"function: retrieval_relevant_chuks(question, top_k = 3) ")

In [ ]:
question = "What is ETL and how does it work in data engineering?"
results = retrieve_relevant_chunks(question, top_k=3)

print(f"Question: {question}\n")
print(f"Found {len(results['documents'][0])} relevant chunks.")
print("---")

# We use zip() to pair each document with its distance and metadata
for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResult {i+1}:")
    print(f"Subject: {meta['subject']}")
    print(f"Topic: {meta['topic']}")
    print(f"Distance: {dist:.4f}")
    # Distance: lower value means more similar to our question
    # ChromaDB uses L2 distance by default (Euclidean distance)
    print(f"Content: {doc[:120]}...") # We show only the first 120 characters to keep output readable
    print("---")


Question: What is ETL and how does it work in data engineering?

Found 3 relevant chunks.
---

Result 1:
Subject: Data Engineering
Topic: ETL Pipelines
Distance: 0.2269
Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...
---

Result 2:
Subject: Data Engineering
Topic: APIs and Data Collection
Distance: 1.0690
Content: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...
---

Result 3:
Subject: Python Programming
Topic: Data Visualization
Distance: 1.3375
Content: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...
---


In [ ]:
def build_context_from_results(results):

    context_parts = []

    for i, (doc, meta) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    )):

        chunk_text = f"Source {i+1}: {meta['subject']} ({meta['topic']})\n{doc}"
        context_parts.append(chunk_text)

    context_str = "\n\n---\n\n".join(context_parts)
    return context_str

In [ ]:
def generate_rag_answer(question, context):

    system_prompt = "You are a helpful academic assistant for engineering students. Only use the provided context to answer the question. If the answer is not in the context, politely say that you don't have enough information."

    client = Groq(
        api_key=os.environ.get("GROQ_API_KEY"),
    )

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {question}",
            },
        ],
        model="llama3-8b-8192", # Using a smaller, faster model for quick responses
        temperature=0.0,
    )

    return chat_completion.choices[0].message.content

